<div align="center">

# Portfolio Optimization 
<br>

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qci-wdyk/eqc-models-tutorial/blob/main/tutorial07-portfolio-optimization.ipynb)

</div>


Select a value for $\xi$ and solve the program
$$
\begin{equation}
    \min_{\{w_{i}\}_{i \in \{1, 2,..., K\}}} [-E(R) R_{B} + \xi \mathrm{VAR}(R)]
\end{equation}
$$

to select a weighted proportion of the portfolio to seek maximum return and minimized risk.


In [6]:
try:
    import eqc_models
except ImportError:
    !pip install eqc-models
import os
import sys
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from eqc_models.allocation import PortMomentum
from eqc_models.solvers import Dirac3ContinuousCloudSolver
try:
    from google.colab import userdata
except ImportError:
    userdata = None

In [ ]:
# Define the API URL and token  for QCI
if userdata is None:
    api_token = "" # replace or use environment variables to configure
    api_url = "https://api.qci-prod.com"
else:
    api_token = userdata.get("QCI_TOKEN")
    api_url = userdata.get("QCI_API_URL")
    os.environ["QCI_TOKEN"] = api_token
    os.environ["QCI_API_URL"] = api_url
    # get some local files
    !mkdir stock_prices/
    !git clone https://github.com/qci-wdyk/eqc-models-tutorial.git
    !cp eqc-models-tutorial/stock_prices/* stock_prices/
    !cp eqc-models-tutorial/nasdaq100_stocks_table.csv nasdaq100_stocks_table.csv
    !cp eqc-models-tutorial/utils.py utils.py

In [8]:
from utils import (
    get_nasdaq100_constituents,
    get_port_stats,
)

In [9]:
# Set parameters
ADJ_DATE = "2022-01-01"
LOOKBACK_DAYS = 60
LOOKFORWARD_DAYS = 30

# Get stock list
stocks = get_nasdaq100_constituents(
    ADJ_DATE, LOOKBACK_DAYS, LOOKFORWARD_DAYS,
)

# Get portfolio model
model = PortMomentum(
    stocks=stocks,
    adj_date=ADJ_DATE,
    stock_data_dir="stock_prices",
    lookback_days=LOOKBACK_DAYS,
    window_days=30,
    window_overlap_days=15,
    weight_upper_limit=0.08,
    r_base=0.05 / 365,
    alpha=5.0, # penalty multiplier term
    beta=1.0, # penalty multiplier term
    xi=1.0, # multiplier as in the formula
)

n = model.n
model.constraints = (
    np.zeros((0, n), dtype=np.float32),
    np.zeros(0, dtype=np.float32),
)

Chose 101 of 102 stocks


In [10]:
# Solve on Dirac-3
solver = Dirac3ContinuousCloudSolver(api_url,api_token)

response = solver.solve(
    model,
    sum_constraint=100,
    relaxation_schedule=4,
    num_samples=10,
)


sol = response["results"]["solutions"][0][:len(stocks)]

print(response)

weight_hash = {}
for i in range(len(stocks)):
    weight_hash[stocks[i]] = sol[i] / 100.0

tot_weight = sum(weight_hash.values())

if tot_weight != 1.0:
    for stock in stocks:
        weight_hash[stock] = weight_hash[stock] / tot_weight

weight_df = pd.DataFrame(
    {
        "Stock": [item for item in weight_hash.keys()],
        "Allocation": [
            weight_hash[item] for item in weight_hash.keys()
        ],
    }
)
weight_df["Date"] = ADJ_DATE
weight_df = weight_df[weight_df["Allocation"] > 0]

ret_df = get_port_stats(weight_df, 30)

print(ret_df)


2026-09-10 15:43:30 - Dirac allocation balance = 0 s (unmetered)
2026-09-10 15:43:30 - Job submitted: job_id='6aa3086308442f441bbb6f0c'
2026-09-10 15:43:30 - QUEUED
2026-09-10 15:43:32 - RUNNING
2026-09-10 15:44:53 - COMPLETED
2026-09-10 15:44:56 - Dirac allocation balance = 0 s (unmetered)
SolutionResults(solutions=array([[0.9963651, 0.9968899, 0.982744 , ..., 0.       , 0.       ,
        0.       ],
       [1.0188671, 0.9865358, 0.9058692, ..., 0.       , 0.       ,
        0.       ],
       [0.9948213, 1.0481919, 0.970878 , ..., 0.       , 0.       ,
        0.       ],
       ...,
       [1.0169253, 1.0303292, 0.9486453, ..., 0.       , 0.       ,
        0.       ],
       [0.9539294, 0.9435859, 0.9309684, ..., 0.       , 0.       ,
        0.       ],
       [1.01033  , 1.0840192, 1.0375237, ..., 0.       , 0.       ,
        0.       ]], shape=(10, 202)), energies=array([-51499.3359375, -51499.3359375, -51499.3320312, -51499.3164062,
       -51498.3828125, -51498.3515625, -514